# Guía de práctica: predicción de la puntuación final de un videojuego

## Escenario

Un estudio dispone de métricas tempranas de varios videojuegos y desea estimar la puntuación crítica final (`final_critic_score`) cuando concluya el período de reseñas. Cada fila representa un videojuego.

Se elige **puntuación crítica** y no “éxito comercial” porque son objetivos distintos. Las ventas dependerían también de distribución, región, plataforma y competencia. En este ejercicio la variable objetivo es continua, por lo que corresponde aplicar regresión.

## Diccionario de columnas

| Columna | Tipo/unidad | Significado | Papel inicial |
|---|---|---|---|
| `game_id` | Texto | Identificador anónimo del juego | Identificador; no entra al modelo |
| `genre` | Categoría | Género principal | Contexto; requeriría codificación para modelar |
| `price_usd` | USD | Precio de lanzamiento | Predictor numérico potencial |
| `development_years` | Años | Duración del desarrollo | Predictor numérico potencial |
| `technical_quality_score` | 1–10 | Estabilidad, rendimiento y acabado técnico | Predictor numérico potencial |
| `early_user_score` | 1–10 | Puntuación media de usuarios durante una etapa temprana | Predictor numérico potencial |
| `completion_rate_pct` | 0–100 % | Porcentaje de usuarios que completó el juego | Predictor numérico potencial |
| `average_playtime_hours` | Horas | Duración media observada | Predictor numérico potencial |
| `marketing_budget_musd` | Millones USD | Presupuesto estimado de mercadotecnia | Predictor numérico potencial |
| `final_critic_score` | 0–100 | Puntuación crítica final | **Variable objetivo (`y`)** |

El conjunto es sintético y reproducible. Las asociaciones encontradas no demuestran causalidad. En un proyecto real debe comprobarse que las métricas tempranas estén disponibles en el momento exacto de la predicción para evitar fuga de información.

## Forma de utilizar la guía

Cada sección contiene:

1. **Documentación:** qué hace cada función y su sintaxis básica.
2. **Objetivo:** resultado que debe obtenerse.
3. **Celda de código:** espacio con instrucciones `TODO` para escribir la solución.
4. **Interpretación:** preguntas que deben responderse después de ejecutar.

No se recomienda copiar código sin comprender los objetos que recibe y devuelve. Ejecute las celdas en orden, porque cada sección utiliza variables creadas anteriormente.

## 1. Cargar y explorar los datos

### Funciones necesarias

- `import pandas as pd`: carga pandas con el alias convencional `pd`.
- `pd.read_csv("archivo.csv")`: lee un CSV y devuelve un `DataFrame`.
- `df.shape`: tupla `(filas, columnas)`.
- `df.head(n)`: primeras `n` filas.
- `df.info()`: tipos de datos, valores no nulos y uso de memoria.
- `df.describe(include="all")`: resumen de columnas numéricas y categóricas.
- `df.columns`: nombres de todas las columnas.
- `df.dtypes`: tipo de cada columna.

### Objetivo

Cargue `videogame_critic_scores.csv` en una variable llamada `df`. Confirme que contiene 520 filas y 10 columnas. Revise si las escalas observadas coinciden con el diccionario.

In [ ]:
# TODO 1.1 — Importe pandas como pd y numpy como np.
# TODO 1.2 — Importe matplotlib.pyplot como plt.
# TODO 1.3 — Lea videogame_critic_scores.csv y guárdelo en df.
# TODO 1.4 — Imprima df.shape, muestre df.head(10), df.info() y df.describe(include="all").

# df = ...

### Preguntas de la sección 1

1. ¿Cuál es la unidad de observación?
2. ¿Qué columnas son identificadores, categorías, predictores numéricos y objetivo?
3. ¿Cuál es el mínimo, máximo y promedio de `final_critic_score`?
4. ¿Hay valores ausentes según `info()`?
5. ¿Por qué `game_id` no debe tratarse como una magnitud numérica?

**Respuesta:** escriba aquí después de ejecutar la celda.

## 2. Correlación cruzada y gráficos de dispersión

### Funciones necesarias

- `df.select_dtypes(include="number")`: selecciona únicamente columnas numéricas.
- `df[columns].corr()`: matriz de correlación de Pearson, con valores entre −1 y 1.
- `plt.imshow(matrix, vmin=-1, vmax=1, cmap="coolwarm")`: representa la matriz con colores.
- `plt.colorbar()`: añade la escala de colores.
- `plt.xticks(...)` / `plt.yticks(...)`: nombres de variables en los ejes.
- `plt.subplots(filas, columnas)`: crea una cuadrícula de gráficos.
- `ax.scatter(x, y)`: diagrama de dispersión.
- `np.polyfit(x, y, 1)`: pendiente e intercepto de una recta descriptiva.
- `np.linspace(min, max, n)`: valores ordenados para dibujar la recta.

### Interpretación

Pearson mide asociación **lineal**. Un valor próximo a +1 indica que ambas variables tienden a aumentar juntas; próximo a −1, que cambian en direcciones opuestas; próximo a 0, que no existe una recta clara. Correlación no implica causalidad.

### Objetivo

Construya una matriz de correlación y un gráfico de dispersión de cada predictor numérico frente a `final_critic_score`. La recta superpuesta permite comparar dirección y dispersión.

In [ ]:
# TODO 2.1 — Cree numeric_columns con select_dtypes(include="number").columns.
# TODO 2.2 — Calcule correlation_matrix = df[numeric_columns].corr().
# TODO 2.3 — Dibuje la matriz con imshow, etiquetas y colorbar.
# TODO 2.4 — Cree una lista de predictores numéricos excluyendo final_critic_score.
# TODO 2.5 — Cree subplots y, mediante un for, dibuje cada predictor vs final_critic_score.
# TODO 2.6 — Opcional: añada una recta con np.polyfit y np.linspace.

# correlation_matrix = ...

### Preguntas de la sección 2

1. ¿Qué tres columnas presentan la mayor correlación absoluta con el objetivo?
2. ¿Qué columnas tienen correlación próxima a cero?
3. ¿Qué gráfico muestra la nube más próxima a una recta?
4. ¿`price_usd` o `marketing_budget_musd` permiten explicar directamente la calidad crítica?
5. ¿Una correlación alta de `early_user_score` demuestra que las reseñas de usuarios causan las reseñas críticas?

**Respuesta:** escriba aquí con valores numéricos.

## 3. Mostrar registros según la correlación observada

### Funciones necesarias

- `correlation_matrix[TARGET].drop(TARGET)`: correlación de cada predictor con el objetivo, sin la autocorrelación 1.0.
- `sort_values(key=abs, ascending=False)`: ordena por magnitud absoluta.
- `.index[0]`: nombre de la primera variable ordenada.
- `df.nlargest(n, column)` / `df.nsmallest(n, column)`: filas con valores mayores o menores.
- `df.loc[:, columns]`: selecciona columnas para una tabla legible.
- `pd.qcut(series, q=4, labels=...)`: divide una variable en cuatro grupos con tamaños similares.
- `groupby(...).agg(...)`: compara el objetivo entre grupos.

### Objetivo

Identifique automáticamente el predictor de mayor correlación absoluta. Muestre cinco juegos con valores altos y cinco con valores bajos de ese predictor. Divida además el predictor en cuartiles y compare la puntuación crítica media.

In [ ]:
TARGET = "final_critic_score"

# TODO 3.1 — Obtenga target_correlations y ordénelo por valor absoluto.
# TODO 3.2 — Guarde el nombre del primer predictor en most_relevant.
# TODO 3.3 — Muestre game_id, most_relevant y TARGET para los 5 valores mayores.
# TODO 3.4 — Repita para los 5 valores menores.
# TODO 3.5 — Cree correlation_group con pd.qcut(..., q=4).
# TODO 3.6 — Agrupe por correlation_group y calcule cantidad, media y mediana de TARGET.

# most_relevant = ...

### Preguntas de la sección 3

1. ¿Cómo cambia `final_critic_score` desde el cuartil bajo al alto?
2. ¿Los cinco registros extremos siguen siempre el patrón general?
3. ¿Por qué una tendencia de grupo no permite predecir perfectamente cada fila?
4. Compare los resultados con una columna de correlación cercana a cero.

**Respuesta:** escriba aquí.

## 4. Separar características (`X`) y objetivo (`y`)

### Funciones necesarias

- `df.drop(columns=[...])`: devuelve una tabla sin las columnas indicadas.
- `df["column"]`: selecciona una columna como `Series`.
- `df.loc[:, feature_columns]`: selecciona un conjunto explícito de predictores.
- `X.shape` / `y.shape`: verifica dimensiones.

### Decisión metodológica

Para mantener el ejercicio básico, use únicamente predictores **numéricos**. Excluya `game_id`, `genre` y `final_critic_score` de `X`. `genre` podría usarse después mediante codificación one-hot, pero no se convierte en un número arbitrario 1, 2, 3, porque ese orden no existe.

### Objetivo

Construya `X` con los siete predictores numéricos y `y` con `final_critic_score`.

In [ ]:
# TODO 4.1 — Cree feature_columns con las siete columnas numéricas predictoras.
# TODO 4.2 — Cree X = df[feature_columns].copy().
# TODO 4.3 — Cree y = df[TARGET].copy().
# TODO 4.4 — Imprima X.shape y y.shape; muestre X.head() y y.head().

# feature_columns = (...)
# X = ...
# y = ...

## 5. Dividir en entrenamiento y prueba

### Función necesaria

```python
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
```

- `test_size=0.20`: reserva 20 % de los juegos para prueba.
- `random_state=42`: hace reproducible la selección.
- El modelo aprende exclusivamente con `X_train`, `y_train`.
- `X_test`, `y_test` se utilizan después para estimar generalización.

No se debe evaluar solo en entrenamiento: un modelo puede memorizar patrones y fallar en juegos nuevos.

In [ ]:
# TODO 5.1 — Importe train_test_split.
# TODO 5.2 — Divida X e y con test_size=0.20 y random_state=42.
# TODO 5.3 — Imprima las formas de X_train, X_test, y_train y y_test.
# TODO 5.4 — Verifique que entrenamiento + prueba = total de filas.

# X_train, X_test, y_train, y_test = ...

## 6. Entrenar y aplicar la regresión lineal

### Funciones necesarias

- `LinearRegression()`: crea un modelo sin entrenar.
- `model.fit(X_train, y_train)`: estima intercepto y coeficientes minimizando los errores cuadrados.
- `model.predict(X_train)`: predicciones sobre entrenamiento.
- `model.predict(X_test)`: predicciones sobre juegos no vistos.
- `model.intercept_`: valor base estimado.
- `model.coef_`: coeficiente de cada predictor en el mismo orden que `X.columns`.
- `pd.Series(model.coef_, index=X.columns)`: tabla legible de coeficientes.

El modelo múltiple es:

$$\widehat{score}=b_0+b_1x_1+\cdots+b_7x_7$$

Cada coeficiente representa el cambio esperado en la predicción por una unidad adicional de ese predictor, manteniendo constantes los demás. No demuestra causalidad.

In [ ]:
# TODO 6.1 — Importe LinearRegression de sklearn.linear_model.
# TODO 6.2 — Cree model = LinearRegression().
# TODO 6.3 — Entrene con model.fit(X_train, y_train).
# TODO 6.4 — Cree y_train_pred y y_test_pred con predict.
# TODO 6.5 — Muestre intercepto y una Series de coeficientes indexada por X.columns.

# model = ...

## 7. Explicar los resultados mediante métricas

### Funciones y fórmulas

- `mean_absolute_error(y_real, y_pred)`: MAE, error absoluto promedio en puntos de crítica.
- `mean_squared_error(...)`: MSE, promedio del error al cuadrado.
- `np.sqrt(MSE)`: RMSE, vuelve a puntos de crítica y penaliza más los errores grandes.
- `r2_score(y_real, y_pred)`: R², proporción de variabilidad explicada respecto de predecir siempre la media.

$$MAE=\frac{1}{n}\sum|y_i-\hat y_i|$$

$$RMSE=\sqrt{\frac{1}{n}\sum(y_i-\hat y_i)^2}$$

$$R^2=1-\frac{\sum(y_i-\hat y_i)^2}{\sum(y_i-\bar y)^2}$$

Menor MAE/RMSE es mejor; mayor R² es mejor. Compare entrenamiento y prueba: una brecha grande puede indicar sobreajuste.

In [ ]:
# TODO 7.1 — Importe mean_absolute_error, mean_squared_error y r2_score.
# TODO 7.2 — Cree una función metrics(y_real, y_pred) que devuelva MAE, RMSE y R2.
# TODO 7.3 — Calcule train_metrics y test_metrics.
# TODO 7.4 — Construya un DataFrame con una fila para entrenamiento y otra para prueba.
# TODO 7.5 — Interprete la brecha entre ambos segmentos.

# metrics_table = ...

### Preguntas de la sección 7

1. Interprete el MAE de prueba en puntos de `final_critic_score`.
2. ¿Por qué RMSE suele ser mayor que MAE?
3. ¿Qué porcentaje de variabilidad explica R² en prueba?
4. ¿Existe una diferencia preocupante entre entrenamiento y prueba?
5. ¿Un R² alto significa que el modelo encontró las causas de una buena reseña?

**Respuesta:** escriba aquí.

## 8. Comparar 20 valores reales y predichos

### Funciones necesarias

- `X_test.iloc[:20].copy()`: copia los primeros 20 registros de prueba.
- `y_test.iloc[:20].to_numpy()`: extrae los objetivos en el mismo orden, sin índice.
- `predictions[:20]`: primeras 20 predicciones.
- `table["error"] = real - predicted`: error con signo.
- `.abs()`: magnitud del error.
- Error porcentual: `abs_error / abs(real) * 100`. Es inestable cuando el valor real es cercano a cero.
- `reset_index()`: conserva el índice original y vuelve a numerar la tabla.
- `display(table)`: presentación tabular en Jupyter.

### Objetivo

Construya tablas separadas de 20 juegos de entrenamiento y 20 de prueba con predictores, `score_real`, `score_predicho`, `error`, `error_absoluto` y `error_porcentual`. La comparación de prueba es la más importante.

In [ ]:
# TODO 8.1 — Escriba una función comparison_table(X_part, y_part, predictions, n=20).
# Dentro de la función:
#   a) copie X_part.iloc[:n]
#   b) agregue score_real y score_predicho
#   c) calcule error, error_absoluto y error_porcentual
#   d) devuelva la tabla con reset_index()
# TODO 8.2 — Cree train_comparison usando X_train/y_train/y_train_pred.
# TODO 8.3 — Cree test_comparison usando X_test/y_test/y_test_pred.
# TODO 8.4 — Muestre ambas tablas y compare el promedio de error_absoluto.

# def comparison_table(...):
#     ...

## 9. Predecir tres videojuegos nuevos

Los tres casos permiten comprobar la interpretación de la correlación:

1. **Señal favorable:** valores altos en las variables más relacionadas con el objetivo.
2. **Señal desfavorable:** valores bajos en esas mismas variables.
3. **Escenario opuesto:** valores medios en las variables relevantes, pero un valor extremo en una variable casi sin correlación (`price_usd` o `marketing_budget_musd`). El objetivo es observar que esa variable por sí sola no domina la predicción.

### Funciones necesarias

- Una lista de diccionarios: cada diccionario representa un juego.
- `pd.DataFrame(records)`: convierte los diccionarios en tabla.
- `new_games.loc[:, feature_columns]`: asegura mismo orden y columnas que el entrenamiento.
- `model.predict(...)`: genera las tres predicciones.
- `assign(predicted_score=...)`: agrega el resultado.

Una predicción con valores fuera del rango de entrenamiento es extrapolación y debe interpretarse con cautela.

In [ ]:
# Los valores están preparados; complete la conversión y la predicción.
new_records = [
    {
        "scenario": "señal favorable",
        "price_usd": 59.99,
        "development_years": 4.5,
        "technical_quality_score": 9.2,
        "early_user_score": 9.1,
        "completion_rate_pct": 88.0,
        "average_playtime_hours": 38.0,
        "marketing_budget_musd": 25.0,
    },
    {
        "scenario": "señal desfavorable",
        "price_usd": 59.99,
        "development_years": 2.0,
        "technical_quality_score": 4.2,
        "early_user_score": 3.8,
        "completion_rate_pct": 32.0,
        "average_playtime_hours": 9.0,
        "marketing_budget_musd": 25.0,
    },
    {
        "scenario": "precio extremo, señal media",
        "price_usd": 129.99,
        "development_years": 3.2,
        "technical_quality_score": 7.0,
        "early_user_score": 6.8,
        "completion_rate_pct": 58.0,
        "average_playtime_hours": 22.0,
        "marketing_budget_musd": 25.0,
    },
]

# TODO 9.1 — new_games = pd.DataFrame(new_records).
# TODO 9.2 — prediction_input = new_games[feature_columns].
# TODO 9.3 — new_games["predicted_final_critic_score"] = model.predict(prediction_input).
# TODO 9.4 — Muestre new_games.

# new_games = ...

## 10. Evaluación final del aprendizaje

Responda con referencia a sus resultados:

1. Explique la diferencia entre `X` e `y`.
2. ¿Por qué se excluyeron `game_id` y `genre` del modelo básico?
3. ¿Cuál fue el predictor de mayor correlación absoluta y qué significa su signo?
4. ¿Por qué una variable de correlación cercana a cero puede tener un coeficiente distinto de cero en la regresión múltiple?
5. ¿Qué función separó entrenamiento y prueba? ¿Por qué se fijó `random_state`?
6. ¿Qué hace `fit()` y qué hace `predict()`?
7. Interprete MAE, RMSE y R² en una frase cada uno.
8. Compare los errores observados en las 20 filas de entrenamiento y las 20 de prueba.
9. ¿Cuál de los tres juegos nuevos recibió mayor puntuación y por qué?
10. ¿El precio extremo produjo por sí solo una predicción extrema? Relaciónelo con la correlación observada.
11. ¿Qué ocurriría si `early_user_score` se recopilara después de conocer la crítica final?
12. ¿Qué variables omitidas podrían mejorar el modelo (género codificado, estudio, plataforma, errores técnicos, innovación, etc.)?
13. ¿Por qué este modelo no debe utilizarse para afirmar que una característica causa una buena reseña?
14. ¿Qué limitación presenta una recta ante relaciones con umbrales, curvas o interacciones?

## Resumen de funciones utilizadas

| Etapa | Funciones principales |
|---|---|
| Carga e inspección | `read_csv`, `shape`, `head`, `info`, `describe`, `dtypes` |
| Correlación y gráficos | `select_dtypes`, `corr`, `imshow`, `subplots`, `scatter`, `polyfit` |
| Selección de casos | `sort_values`, `nlargest`, `nsmallest`, `loc`, `qcut`, `groupby`, `agg` |
| Preparación | `drop`, selección con lista/tupla, `copy` |
| Partición | `train_test_split` |
| Entrenamiento | `LinearRegression`, `fit`, `predict`, `coef_`, `intercept_` |
| Evaluación | `mean_absolute_error`, `mean_squared_error`, `sqrt`, `r2_score` |
| Comparación | `iloc`, `to_numpy`, `abs`, `reset_index`, `display` |
| Casos nuevos | lista de diccionarios, `DataFrame`, `assign`, `predict` |

## Lista de comprobación

- [ ] Dataset explorado y columnas interpretadas.
- [ ] Correlaciones y gráficos construidos.
- [ ] Registros extremos comparados.
- [ ] `X` e `y` separados correctamente.
- [ ] Partición reproducible de entrenamiento/prueba.
- [ ] Modelo entrenado y coeficientes revisados.
- [ ] Métricas calculadas para ambos segmentos.
- [ ] Veinte valores reales y predichos comparados.
- [ ] Tres nuevos juegos evaluados.
- [ ] Preguntas respondidas con evidencia, sin afirmar causalidad.